### Преобразуем орбитальные элементы в вектор состояния (считаем эфемериды)

In [1]:
from math import *
import numpy as np

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u

from datetime import datetime

from astroquery.jplhorizons import Horizons

from astroquery.jplsbdb import SBDB
ast = '2014 HK129'

In [2]:
# запрос данных для астероида в базе данных тел Солнечной системы
sbdb = SBDB.query(ast)

In [3]:
# визуализация результата запроса
sbdb

OrderedDict([('object',
              OrderedDict([('orbit_id', '50'),
                           ('neo', True),
                           ('pha', True),
                           ('des', '2014 HK129'),
                           ('kind', 'au'),
                           ('orbit_class',
                            OrderedDict([('name', 'Apollo'),
                                         ('code', 'APO')])),
                           ('prefix', None),
                           ('fullname', '(2014 HK129)'),
                           ('spkid', '3669359')])),
             ('signature',
              OrderedDict([('source',
                            'NASA/JPL Small-Body Database (SBDB) API'),
                           ('version', '1.3')])),
             ('orbit',
              OrderedDict([('last_obs', '2022-12-20'),
                           ('n_obs_used', 270),
                           ('source', 'JPL'),
                           ('moid', <Quantity 0.00863 AU>),
              

In [4]:
# отбираем орбитальные элементы
a = sbdb['orbit']['elements']['a'].value
e = sbdb['orbit']['elements']['e']
T = sbdb['orbit']['elements']['tp'].value
P = sbdb['orbit']['elements']['per'].value

ep0 = sbdb['orbit']['cov_epoch'].value

# текущее время
ut = Time(datetime.utcnow(), scale='utc')
ep = ut.jd
ep_tdb = ut.tdb.jd

# ep,ep_tdb,P,T

In [5]:
e

0.489

In [6]:
# Решение уравнения Кеплера
def KeplerEquation(e,T,P,t):
    n = 2*pi/P
    E = M = n*(t-T)
    epsilon = radians(1/36000000.0)
    while(fabs(E-e*sin(E)-M)>epsilon):
        E = M+e*sin(E)
    return E

# Орбитальные координаты и скорости
def orbitXY(a,e,T,P,t):
    E = KeplerEquation(e, T, P, t)
    x = cos(E)-e
    y = sqrt(1-e**2)*sin(E)
    dotE = 2 * pi / P /(1-e*cos(E))
    dotx = -sin(E)*dotE
    doty = sqrt(1-e**2)*cos(E)*dotE
    return np.array([a*x,a*y,0]),np.array([a*dotx,a*doty,0])

In [7]:
# вычисляем орбитальные координаты и скорости
r,v = orbitXY(a,e,T,P,ep)
r,v

(array([-1.88365601,  1.16460399,  0.        ]),
 array([-0.00795003, -0.00546599,  0.        ]))

In [8]:
# матрицы вращения
def R_z(theta):
    return np.array([[cos(radians(theta)),sin(radians(theta)),0],\
                     [-sin(radians(theta)),cos(radians(theta)),0],\
                     [0,0,1]])
def R_x(theta):
    return np.array([[1,0,0],\
                 [0,cos(radians(theta)),sin(radians(theta))],\
                 [0,-sin(radians(theta)),cos(radians(theta))]])

In [9]:
# параметры ориентации орбиты
w = sbdb['orbit']['elements']['w'].value
i = sbdb['orbit']['elements']['i'].value
om = sbdb['orbit']['elements']['om'].value

In [10]:
# матрица поворота
M = np.dot(R_z(-om),np.dot(R_x(-i),R_z(-w)))

In [11]:
# вычисление вектора состояния в эклиптической системе координат
r = np.dot(M,r)
v = np.dot(M,v)
r,v

(array([ 0.65770202, -2.11381237, -0.06072684]),
 array([ 0.00952296, -0.00107803, -0.00110951]))

In [12]:
# обращение к Horizons
obj = Horizons(id=ast, id_type='smallbody', location='@0', epochs=ep_tdb)
# obj = Horizons(id=ast, id_type='smallbody', location='@sun', epochs=ep_tdb)
# вектор состояния (опорная плоскость - средний экватор на эпоху J2000)
pos_ast = obj.vectors(refplane='earth')
pos_ast 

targetname,datetime_jd,datetime_str,H,G,x,y,z,vx,vy,vz,lighttime,range,range_rate
---,d,---,mag,---,AU,AU,AU,AU / d,AU / d,AU / d,d,AU,AU / d
str12,float64,str30,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
(2014 HK129),2460209.89212695,A.D. 2023-Sep-22 09:24:39.7685,21.1,0.15,0.652220303749988,-1.91654148577048,-0.8970725914464591,0.00953308764521464,-0.0005401378955399539,-0.001444234579008683,0.01278890934505384,2.214331010853507,0.00386051282261958


In [13]:
# координаты
c = SkyCoord(pos_ast['x'][0], pos_ast['y'][0],pos_ast['z'][0], representation_type='cartesian', frame='icrs')
# скорости
cv = SkyCoord(pos_ast['vx'][0], pos_ast['vy'][0],pos_ast['vz'][0], representation_type='cartesian', frame='icrs')
# переход к эклиптике
c_ecl = c.transform_to('barycentricmeanecliptic')
cv_ecl = cv.transform_to('barycentricmeanecliptic')
# переход к более удобной форме
r_ecl = np.array([c_ecl.cartesian.x,c_ecl.cartesian.y,c_ecl.cartesian.z])
v_ecl = np.array([cv_ecl.cartesian.x,cv_ecl.cartesian.y,cv_ecl.cartesian.z])

In [14]:
# разности координат (наша эфемерида минус NASA JPL)
r-r_ecl

array([ 5.48165217e-03,  1.41501964e-03, -3.48206643e-05])

In [15]:
# разности скоростей (наша эфемерида минус NASA JPL)
v-v_ecl

array([-1.01279819e-05, -7.98230741e-06,  6.99002183e-07])

In [16]:
# Задание
# Вычислить наборы координат и скоростей, задав нормальное распределение 
# орбитальных элементов согласно известным стандартным ошибкам 'a_sig'.
# Построить распределение точек в проекции на плоскость x,y.
# Будет ли распределение компонент вектора состояния тоже нормальным?

In [19]:
e_sig = sbdb['orbit']['elements']['e_sig']
ecc = np.random.normal(e,e_sig,100)
ecc

array([0.48899999, 0.48900001, 0.48899995, 0.489     , 0.48899994,
       0.48900003, 0.48900003, 0.48900002, 0.489     , 0.48900001,
       0.489     , 0.48900001, 0.48900002, 0.48899997, 0.489     ,
       0.48899999, 0.48900005, 0.48900003, 0.48899998, 0.48899994,
       0.48899993, 0.48900003, 0.48899996, 0.48900004, 0.48899997,
       0.48899999, 0.48900001, 0.48900006, 0.48900001, 0.48899999,
       0.48900001, 0.48899999, 0.48900001, 0.48900001, 0.48899995,
       0.48899999, 0.489     , 0.48900004, 0.48899995, 0.48899995,
       0.48900001, 0.48900003, 0.48900001, 0.48900004, 0.48899996,
       0.48899999, 0.48899996, 0.48899998, 0.48899996, 0.48900002,
       0.48900001, 0.48899999, 0.48899995, 0.48900001, 0.48900002,
       0.48899997, 0.48899998, 0.48899996, 0.48900003, 0.48899998,
       0.48899995, 0.48900004, 0.48900003, 0.48899998, 0.48900005,
       0.48899992, 0.48900008, 0.48899996, 0.48899997, 0.48900001,
       0.48900007, 0.48900002, 0.489     , 0.48900004, 0.48900